# Terjemahan Fable 5k ke Bahasa Melayu (Kaggle T4 x2 / P100)

**Panduan**
1. Settings -> Accelerator: **GPU T4 x2**; **Internet: ON** (wajib untuk muat turun model & pip).
2. Buat **Kaggle Dataset private bernama `xcoder-mt`** (New Dataset -> upload):
   - `fable_agent.jsonl` (dari repo: `datasets/fable_agent.jsonl`)
   - `import_fable.py` (dari repo: `datasets/scripts/import_fable.py`)
   - pilihan: `.fable_mt_cache.jsonl` (dari PC `datasets\.fable_mt_cache.jsonl`) untuk sambung dari segmen yang sudah siap.
     Jika upload menolak nama bertitik, guna `fable_mt_cache.jsonl` - notebook terima kedua-dua nama.
3. Jalankan sel dari atas ke bawah. Sel server akan uji terjemahan sendiri sebelum sel terjemahan.
4. Terputus? Run semula dari sel salin fail - cache di `/kaggle/working` disambung automatik.
5. Selesai: `fable_agent_ms.jsonl` di `/kaggle/working` - muat turun dari tab **Output** (Save Version dahulu) atau File browser.
6. Selepas ini di PC: letak fail dalam `datasets/`, jalan `merge_one.py --force`, upload `all.jsonl` baharu sebagai dataset `xcoder-training` (versi baharu), kemudian run notebook `kaggle_qwen35`.

Anggaran: ~2-3 jam di T4. Fail di `/kaggle/working` kekal antara sesi (20GB).

In [ ]:
!nvidia-smi

In [ ]:
import os, shutil

SRC = '/kaggle/input/xcoder-mt'
WORK = '/kaggle/working'
os.makedirs(WORK, exist_ok=True)
if not os.path.isdir(SRC) and not os.path.exists(f'{WORK}/fable_agent.jsonl'):
    raise SystemExit(f'{SRC} tiada - buat Kaggle Dataset "xcoder-mt" dengan '
                     'fable_agent.jsonl + import_fable.py (+ .fable_mt_cache.jsonl).')
for name in ('fable_agent.jsonl', 'import_fable.py'):
    if not os.path.exists(f'{WORK}/{name}'):
        shutil.copy(f'{SRC}/{name}', f'{WORK}/{name}')
    print(name, round(os.path.getsize(f'{WORK}/{name}') / 1e6, 1), 'MB')

cache_dst = f'{WORK}/.fable_mt_cache.jsonl'
if not os.path.exists(cache_dst):
    for cname in ('.fable_mt_cache.jsonl', 'fable_mt_cache.jsonl'):
        if os.path.exists(f'{SRC}/{cname}'):
            shutil.copy(f'{SRC}/{cname}', cache_dst)
            break
if os.path.exists(cache_dst):
    n = sum(1 for l in open(cache_dst, encoding='utf-8') if l.strip())
    print('cache:', n, 'entri')

In [ ]:
# llama.cpp rasmi + --reasoning off (config SAMA yang diuji berjaya lokal).
import json, os, stat, urllib.request, zipfile

rel = json.load(urllib.request.urlopen(
    'https://api.github.com/repos/ggml-org/llama.cpp/releases/latest'))
print('release:', rel['tag_name'])
assets = [a['browser_download_url'] for a in rel['assets']
          if 'ubuntu' in a['name'].lower() and 'cuda' in a['name'].lower()
          and a['name'].endswith('.zip')]
if not assets:
    assets = [a['browser_download_url'] for a in rel['assets']
              if a['name'].endswith('.zip') and 'ubuntu' in a['name'].lower()]
print('asset:', assets[0])
urllib.request.urlretrieve(assets[0], '/kaggle/working/llama.zip')
with zipfile.ZipFile('/kaggle/working/llama.zip') as z:
    z.extractall('/kaggle/working/llama-cpp')
os.remove('/kaggle/working/llama.zip')
SERVER_BIN = None
for root, _, files in os.walk('/kaggle/working/llama-cpp'):
    for f in files:
        if f == 'llama-server':
            SERVER_BIN = os.path.join(root, f)
os.chmod(SERVER_BIN, os.stat(SERVER_BIN).st_mode | stat.S_IEXEC)
os.environ['LD_LIBRARY_PATH'] = os.path.dirname(SERVER_BIN)
print('SERVER_BIN =', SERVER_BIN)

In [ ]:
import os, subprocess, time, requests

if not os.path.exists('/kaggle/working/model.gguf'):
    from huggingface_hub import hf_hub_download
    p = hf_hub_download('unsloth/gemma-4-E4B-it-GGUF',
                        'gemma-4-E4B-it-Q4_K_M.gguf',
                        local_dir='/kaggle/working')
    os.replace(p, '/kaggle/working/model.gguf')
print('model sedia:', round(os.path.getsize('/kaggle/working/model.gguf') / 1e9, 2), 'GB')

srv = subprocess.Popen(
    [SERVER_BIN, '-m', '/kaggle/working/model.gguf',
     '--host', '127.0.0.1', '--port', '8080', '--alias', 'gemma',
     '-ngl', '99', '-c', '8192', '--reasoning', 'off'],
    stdout=open('/kaggle/working/server.log', 'w'), stderr=subprocess.STDOUT)

ok = False
for i in range(150):
    time.sleep(2)
    try:
        if requests.get('http://127.0.0.1:8080/v1/models', timeout=3).status_code == 200:
            ok = True
            print('server sedia selepas', (i + 1) * 2, 's')
            break
    except Exception:
        pass
if not ok:
    print('server gagal - lihat /kaggle/working/server.log')
    print(open('/kaggle/working/server.log').read()[-3000:])
else:
    r = requests.post('http://127.0.0.1:8080/v1/chat/completions',
                      json={'model': 'gemma',
                            'messages': [{'role': 'user',
                                          'content': 'Terjemah ke Bahasa Melayu: hello world'}],
                            'max_tokens': 32}, timeout=120)
    print('uji:', r.status_code, r.json()['choices'][0]['message']['content'].strip()[:80])

In [ ]:
!python /kaggle/working/import_fable.py translate \
  --in /kaggle/working/fable_agent.jsonl \
  --out /kaggle/working/fable_agent_ms.jsonl \
  --server http://127.0.0.1:8080 --model gemma --workers 2 \
  --translate-output --verbose \
  --cache /kaggle/working/.fable_mt_cache.jsonl

In [ ]:
!python /kaggle/working/import_fable.py clean \
  --in /kaggle/working/fable_agent_ms.jsonl \
  --out /kaggle/working/fable_agent_ms.jsonl
print('Siap - muat turun fable_agent_ms.jsonl dari tab Output / File browser')